In [ ]:
# =====================================================================
# Pareto Jumps — Two-Jump Analysis
#   Jump A @ 6.1%: Lz 0.65→0.26 (−60%)  TRUE 十字転置
#   Jump B @ 9.5%: Lx 1.99→0.17 (−91%)  同向极限压缩
# =====================================================================

import torch, numpy as np, math
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.termination import get_termination
from pymoo.optimize import minimize

device=torch.device('cuda'); print(f'Device: {device}')

# ============================================================
# Physical Model
# ============================================================
rx,ry,rz=10.0,10.0,3.0; xBs,yBs,zBs=10.0,-100.0,-10.0
E=8; dB=0.075; lam=0.075; L1=2; dref=abs(yBs)*1.5
Pd=40.0; Rth=0.2; Nd=-174.0; Bw=20e6; pm=0.2; nu=5
PB=10**(Pd/10)*1e-3; N0=10**(Nd/10)*1e-3*Bw
# ==========================================
# RWP Generator (for KDE weights)
# ==========================================
def gen_rwp_traj(sim_time,dt=10,nu=5,rx=10.0,ry=10.0,rng=None):
    if rng is None: rng=np.random
    rs=[0.0,rx,0.0,ry]; hc=np.array([rx/2,ry/2]); hr=rx/3.0
    p_t=0.6;p_s=0.3;tau_h=1.5;tau_n=0.1;v_h=0.2;v_n=1.0;g_h=0.6;g_n=0.2
    ts=int(sim_time/dt)
    def gt(p):
        t=hc+(rng.rand(2)-0.5)*2*hr if rng.rand()<g_h else np.array([rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])])
        t[0]=np.clip(t[0],rs[0],rs[1]);t[1]=np.clip(t[1],rs[2],rs[3]);return t
    def ta(r):return tau_h if r=='hot'else tau_n
    def sp(r):return v_h if r=='hot'else v_n
    uh=1.5+0.5*rng.rand(nu);pos=np.zeros((nu,ts,2))
    up=np.zeros((nu,2));ur=[None]*nu;us=[None]*nu;ut_=np.zeros((nu,2));ud=np.zeros((nu,2));usp=np.zeros(nu);uprem=np.zeros(nu)
    for i in range(nu):
        up[i]=[rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])];d2h=np.linalg.norm(up[i]-hc);ur[i]='hot'if d2h<=hr else'normal'
        if rng.rand()<p_t:us[i]='transfer';ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
        else:us[i]='pause';uprem[i]=rng.exponential(ta(ur[i]))
        pos[i,0,:]=up[i]
    for step in range(1,ts):
        for i in range(nu):
            if us[i]=='pause':
                uprem[i]-=dt;pos[i,step,:]=up[i]
                if uprem[i]<=0:us[i]='transfer';ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
            else:
                md=usp[i]*dt;pd=ut_[i]-up[i]
                if np.linalg.norm(pd)<=md:
                    up[i]=ut_[i].copy();d2h=np.linalg.norm(up[i]-hc);ur[i]='hot'if d2h<=hr else'normal'
                    if rng.rand()<p_s:us[i]='pause';uprem[i]=rng.exponential(ta(ur[i]))
                    else:ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
                else:up[i]=up[i]+ud[i]*md
                pos[i,step,:]=up[i]
    pts=np.zeros((nu*ts,3));idx=0
    for u in range(nu):
        for s in range(ts):pts[idx]=[pos[u,s,0],pos[u,s,1],uh[u]];idx+=1
    return pts

# Multi-Height Grid + KDE Weights
GR=200
Z_HEIGHTS=[1.5,1.625,1.75,1.875,2.0]; N_Z=len(Z_HEIGHTS)
xv=torch.linspace(0,rx,GR,device=device); yv=torch.linspace(0,ry,GR,device=device)
Xg,Yg=torch.meshgrid(xv,yv,indexing="ij"); xyf=torch.stack([Xg.flatten(),Yg.flatten()],dim=1)
gl=[]; [gl.append(torch.stack([Xg.flatten(),Yg.flatten(),torch.full_like(Xg.flatten(),zh)],dim=1)) for zh in Z_HEIGHTS]
gl=torch.cat(gl,dim=0); Ngrid=gl.shape[0]
print("Building KDE weights...")
np.random.seed(777); et=gen_rwp_traj(sim_time=100000,dt=10)
kde=gaussian_kde(et[:,:2].T,bw_method="scott")
wxy=kde(xyf.cpu().numpy().T).flatten();wxy=wxy/wxy.sum()
gw=torch.tensor(np.tile(wxy,N_Z),dtype=torch.float32,device=device);gw=gw/gw.sum()
print(f"  Grid: {N_Z}x{GR}x{GR}={Ngrid} pts, hotspot/edge={gw.max().item()/gw.min().item():.1f}x")
np.random.seed(42)
_ps=torch.tensor(2*np.pi*np.random.rand(Ngrid),dtype=torch.float32,device=device)
_tt=torch.tensor(-np.pi+2*np.pi*np.random.rand(Ngrid),dtype=torch.float32,device=device)
_et=torch.tensor(10**((-15+5*np.random.rand(Ngrid))/10),dtype=torch.float32,device=device)

def phys_chan(wp,lc):
    if not isinstance(wp,torch.Tensor): wp=torch.tensor(wp,dtype=torch.float32,device=device)
    if wp.dim()==1: wp=wp.unsqueeze(0)
    Bn=wp.shape[0]; xc,zc,Lx,Lz=wp[:,0],wp[:,1],wp[:,2],wp[:,3]
    xu,yu,zu=lc[:,0],lc[:,1],lc[:,2]
    dx=xc-xBs; dy=torch.full_like(xc,0.0-yBs); dz=zc-zBs
    Rw=torch.sqrt(dx**2+dy**2+dz**2); th=torch.atan2(dy,dx); ph=torch.acos(dz/Rw)
    kx=torch.sin(ph)*torch.cos(th); kz=torch.cos(ph)
    x1=xc-Lx/2;z1=zc-Lz/2;x2=xc-Lx/2;z2=zc+Lz/2;x3=xc+Lx/2;z3=zc-Lz/2;x4=xc+Lx/2;z4=zc+Lz/2
    def rd(xs,zs):
        ddx=xs.unsqueeze(1)-xBs; ddy=torch.full_like(ddx,0.0-yBs); ddz=zs.unsqueeze(1)-zBs
        L=torch.sqrt(ddx**2+ddy**2+ddz**2); return ddx/L,ddy/L,ddz/L
    ux1,_,uz1=rd(x1,z1);ux2,_,uz2=rd(x2,z2);ux3,_,uz3=rd(x3,z3);ux4,_,uz4=rd(x4,z4)
    dux=xu-xBs;duy=yu-yBs;duz=zu-zBs; Lu=torch.sqrt(dux**2+duy**2+duz**2)
    uux=dux/Lu;uuz=duz/Lu
    ua=torch.stack([ux1,ux2,ux3,ux4],0); uz=torch.stack([uz1,uz2,uz3,uz4],0)
    umin=ua.min(0).values;umax=ua.max(0).values;zmin=uz.min(0).values;zmax=uz.max(0).values
    b=1000.0
    ix=torch.sigmoid(b*(uux-umin))*torch.sigmoid(b*(umax-uux))
    iz=torch.sigmoid(b*(uuz-zmin))*torch.sigmoid(b*(zmax-uuz))
    il=ix*iz
    d2x=xu-xc.unsqueeze(1);d2y=yu;d2z=zu-zc.unsqueeze(1)
    Ru=torch.sqrt(d2x**2+d2y**2+d2z**2);t1=d2x/Ru;t2=d2z/Ru
    ax=(1-il)*(kx.unsqueeze(1)-t1);az=(1-il)*(kz.unsqueeze(1)-t2)
    sx=torch.sinc((math.pi/lam)*Lx.unsqueeze(1)*ax);sz=torch.sinc((math.pi/lam)*Lz.unsqueeze(1)*az)
    na=torch.arange(E,dtype=torch.float32,device=device)
    p1=(2*math.pi/lam)*dB*na*torch.sin(th).unsqueeze(1)
    a1=(1/math.sqrt(E))*torch.complex(torch.cos(p1),torch.sin(p1))
    v1m=(lam/(4*math.pi))/Rw;v1p=-(2*math.pi/lam)*Rw
    v1=torch.complex(v1m*torch.cos(v1p),v1m*torch.sin(v1p));H1=v1.unsqueeze(1)*a1.conj()
    p2=(2*math.pi/lam)*dB*na.unsqueeze(0)*torch.sin(_tt).unsqueeze(1)
    a2=(1/math.sqrt(E))*torch.complex(torch.cos(p2),torch.sin(p2))
    v2m=_et*(lam/(4*math.pi*dref));v2=torch.complex(v2m*torch.cos(_ps),v2m*torch.sin(_ps))
    H2=v2.unsqueeze(1)*a2.conj().unsqueeze(0)
    Ht=math.sqrt(E/L1)*(H1.unsqueeze(1)+H2)
    fm=(Lx.unsqueeze(1)*Lz.unsqueeze(1)*sx*sz)/(lam*Ru)
    fp=(-(2*math.pi/lam)*(kx*xc+kz*zc)-(math.pi/2))
    fc=torch.complex(fm*torch.cos(fp.unsqueeze(1)),fm*torch.sin(fp.unsqueeze(1)))
    return Ht*fc.unsqueeze(2)

@torch.no_grad()
def eval_phys(X,batch=200):
    out=[]
    for i in range(0,len(X),batch):
        xb=X[i:i+batch]; th=torch.tensor(xb,dtype=torch.float32,device=device)
        He=phys_chan(th,gl); Hw=torch.sum(He,dim=2)/math.sqrt(E)
        dp=(torch.abs(Hw)**2)*pm*PB; it=(nu-1)*dp; sr=dp/(it+N0)
        # Self-blockage (Ref C.3): 4-direction E[Se]
        xc_,zc_=th[:,0],th[:,1]; ab=math.pi/3.0; Ses=0
        gx,gy,gz=gl[:,0].unsqueeze(0),gl[:,1].unsqueeze(0),gl[:,2].unsqueeze(0)
        rn=torch.sqrt((gx-xc_.unsqueeze(1))**2+(gz-zc_.unsqueeze(1))**2+gy**2+1e-12)
        for a in [0.0,math.pi/2,math.pi,3*math.pi/2]:
            dot=torch.abs((gx-xc_.unsqueeze(1))*math.cos(a)+gy*math.sin(a))
            cp=dot/rn; ph=torch.acos(torch.clamp(cp,0,1))
            Se=(math.pi-torch.clamp(2*ab-ph,min=0))/math.pi
            Ses=Ses+torch.clamp(Se,1/3,1)
        sr=((Ses/4)*dp)/((Ses/4)*it+N0)
        rt=torch.log2(1+sr); bits=(rt<Rth).float()
        out.append(torch.sum(bits*gw,dim=1).cpu().numpy())
    return np.concatenate(out)

print(f'Model ready: {Ngrid} points')

# ============================================================
# NSGA-II
# ============================================================
lb=np.array([0.2,0.2,0.1,0.1]); ub=np.array([9.8,2.8,9.8,2.8])
class DP(ElementwiseProblem):
    def __init__(self): super().__init__(n_var=4,n_obj=2,n_ieq_constr=4,xl=lb,xu=ub)
    def _evaluate(self,x,out,*a,**k):
        xc,zc,Lx,Lz=x
        out["G"]=[Lx/2-xc,xc+Lx/2-rx,Lz/2-zc,zc+Lz/2-rz]
        out["F"]=[Lx*Lz,float(eval_phys(x.reshape(1,-1))[0])]

alg=NSGA2(pop_size=300,n_offsprings=150,sampling=FloatRandomSampling(),
    crossover=SBX(prob=0.9,eta=15),mutation=PM(prob=0.25,eta=20),eliminate_duplicates=True)
print('\nNSGA-II (pop=300, gen=200)...')
res=minimize(DP(),alg,get_termination("n_gen",200),seed=42,verbose=True)

F=res.F; X=res.X
def nondom(F):
    d=np.zeros(len(F),dtype=bool)
    for i in range(len(F)):
        for j in range(len(F)):
            if i!=j and F[j,0]<=F[i,0] and F[j,1]<=F[i,1] and (F[j,0]<F[i,0] or F[j,1]<F[i,1]):
                d[i]=True;break
    pf=F[~d]; idx=np.argsort(pf[:,1]); return pf[idx]
pf=nondom(F)
pf_idx=[np.where((F[:,0]==pf[i,0])&(F[:,1]==pf[i,1]))[0][0] for i in range(len(pf))]
pf_X=X[pf_idx]
print(f'Pareto: {len(pf)} points')

# ============================================================
# Locate jumps
# ============================================================
area_ratio=np.ones(len(pf)-1)
for i in range(len(pf)-1):
    area_ratio[i]=max(pf[i,0],pf[i+1,0])/max(min(pf[i,0],pf[i+1,0]),1e-9)
# Jump A: 5-8% outage
jA=np.argmax(area_ratio*((pf[:-1,1]>=0.05)&(pf[:-1,1]<=0.08)))
A_A=pf_X[jA]; B_A=pf_X[jA+1]
# Jump B: 7-12% outage
jB=np.argmax(area_ratio*((pf[:-1,1]>=0.07)&(pf[:-1,1]<=0.12)))
A_B=pf_X[jB]; B_B=pf_X[jB+1]

print(f'\nJump A @ {pf[jA,1]*100:.1f}%: area {pf[jA,0]:.3f}→{pf[jA+1,0]:.3f} m²')
print(f'  A: [{A_A[0]:.3f},{A_A[1]:.3f},{A_A[2]:.3f},{A_A[3]:.3f}] Lz={A_A[3]:.3f}m')
print(f'  B: [{B_A[0]:.3f},{B_A[1]:.3f},{B_A[2]:.3f},{B_A[3]:.3f}] Lz={B_A[3]:.3f}m  ΔLz={B_A[3]-A_A[3]:.3f}m ({abs(B_A[3]-A_A[3])/A_A[3]*100:.0f}%)')
print(f'\nJump B @ {pf[jB,1]*100:.1f}%: area {pf[jB,0]:.3f}→{pf[jB+1,0]:.3f} m²')
print(f'  A: [{A_B[0]:.3f},{A_B[1]:.3f},{A_B[2]:.3f},{A_B[3]:.3f}] Lx={A_B[2]:.3f}m')
print(f'  B: [{B_B[0]:.3f},{B_B[1]:.3f},{B_B[2]:.3f},{B_B[3]:.3f}] Lx={B_B[2]:.3f}m  ΔLx={B_B[2]-A_B[2]:.3f}m ({abs(B_B[2]-A_B[2])/A_B[2]*100:.0f}%)')

# ============================================================
# FIGURE 1 — Decision Tracking
# ============================================================
fig1,axes1=plt.subplots(2,3,figsize=(16,9))
ax=axes1[0,0]
ax.plot(pf[:,1]*100,pf[:,0],'o-',color='darkgreen',markersize=2,linewidth=1)
ax.plot(pf[jA,1]*100,pf[jA,0],'ro',markersize=10,markeredgewidth=2);ax.plot(pf[jA+1,1]*100,pf[jA+1,0],'bD',markersize=8)
ax.plot(pf[jB,1]*100,pf[jB,0],'ro',markersize=10,markeredgewidth=2);ax.plot(pf[jB+1,1]*100,pf[jB+1,0],'bD',markersize=8)
ax.annotate('A',(pf[jA,1]*100,pf[jA,0]),fontsize=11,color='red',fontweight='bold')
ax.annotate('B',(pf[jB,1]*100,pf[jB,0]),fontsize=11,color='red',fontweight='bold')
ax.set_xlabel('Outage [%]');ax.set_ylabel('Area [m²]');ax.set_title('(a) Pareto Front');ax.grid(alpha=0.3)

labs=['xc [m]','zc [m]','Lx [m]','Lz [m]']
for i,(a2,lb) in enumerate(zip([axes1[0,1],axes1[1,0],axes1[1,1],axes1[0,2]],labs)):
    a2.plot(pf[:,1]*100,pf_X[:,i],'o-',color='steelblue',markersize=1.5,linewidth=0.6)
    a2.set_xlabel('Outage [%]');a2.set_ylabel(lb);a2.set_title(f'({chr(98+i)}) {lb}');a2.grid(alpha=0.3)
    a2.axvline(x=pf[jA,1]*100,color='red',ls='--',alpha=0.3)
    a2.axvline(x=pf[jB,1]*100,color='orange',ls='--',alpha=0.3)

ax=axes1[1,2]
asp=pf_X[:,2]/np.maximum(pf_X[:,3],1e-6)
ax.plot(pf[:,1]*100,asp,'o-',color='purple',markersize=1.5,linewidth=0.6)
ax.set_xlabel('Outage [%]');ax.set_ylabel('Lx/Lz');ax.set_title('(f) Aspect Ratio');ax.grid(alpha=0.3)
ax.axvline(x=pf[jA,1]*100,color='red',ls='--',alpha=0.3)
ax.axvline(x=pf[jB,1]*100,color='orange',ls='--',alpha=0.3)
ax.axhline(y=1,color='gray',ls=':',alpha=0.5)
plt.tight_layout();plt.savefig('fig1_tracking.png',dpi=150);plt.show()

# ============================================================
# FIGURE 2 — Jump A landscape + scans
# ============================================================
xcA=(A_A[0]+B_A[0])/2;zcA=(A_A[1]+B_A[1])/2
lx_g=np.linspace(2,10,100);lz_g=np.linspace(0.1,1.5,100)
LX,LZ=np.meshgrid(lx_g,lz_g)
sh_in=np.column_stack([np.full_like(LX.ravel(),xcA),np.full_like(LX.ravel(),zcA),LX.ravel(),LZ.ravel()])
sh_out=eval_phys(sh_in,batch=500).reshape(100,100)

fig2,axes2=plt.subplots(1,3,figsize=(18,5))
ax=axes2[0]
cs=ax.contourf(LX,LZ,sh_out*100,levels=30,cmap='RdYlGn_r')
ax.plot(A_A[2],A_A[3],'ro',markersize=12,markeredgewidth=2,label=f'A (Lx={A_A[2]:.1f},Lz={A_A[3]:.2f})')
ax.plot(B_A[2],B_A[3],'bD',markersize=10,markeredgewidth=2,label=f'B (Lx={B_A[2]:.1f},Lz={B_A[3]:.2f})')
ax.contour(LX,LZ,sh_out*100,levels=[6,7,8,10],colors='white',linewidths=[0.5,0.5,0.5,2],linestyles=[':',':',':','-'])
plt.colorbar(cs,ax=ax,label='Outage [%]');ax.set_xlabel('Lx [m]');ax.set_ylabel('Lz [m]')
ax.set_title(f'Jump A: (Lx,Lz) at xc={xcA:.1f},zc={zcA:.2f}');ax.legend();ax.grid(alpha=0.2)

lx_s=np.linspace(2,10,300)
si=np.column_stack([np.full_like(lx_s,xcA),np.full_like(lx_s,zcA),lx_s,np.full_like(lx_s,(A_A[3]+B_A[3])/2)])
so=eval_phys(si,batch=100)
ax=axes2[1]
ax.plot(lx_s,so*100,'b-',linewidth=1.5);ax.axvline(x=A_A[2],color='red',ls='--')
ax.axvline(x=B_A[2],color='green',ls='--');ax.set_xlabel('Lx [m]');ax.set_ylabel('Outage [%]')
ax.set_title('Lx scan');ax.grid(alpha=0.3)

lz_s=np.linspace(0.1,1.5,300)
si2=np.column_stack([np.full_like(lz_s,xcA),np.full_like(lz_s,zcA),np.full_like(lz_s,(A_A[2]+B_A[2])/2),lz_s])
so2=eval_phys(si2,batch=100)
ax=axes2[2]
ax.plot(lz_s,so2*100,'b-',linewidth=1.5);ax.axvline(x=A_A[3],color='red',ls='--')
ax.axvline(x=B_A[3],color='green',ls='--');ax.set_xlabel('Lz [m]');ax.set_ylabel('Outage [%]')
ax.set_title('Lz scan');ax.grid(alpha=0.3)
plt.tight_layout();plt.savefig('fig2_jumpA.png',dpi=150);plt.show()

# ============================================================
# FIGURE 3 — Jump B landscape + window geometry for both
# ============================================================
xcB=(A_B[0]+B_B[0])/2;zcB=(A_B[1]+B_B[1])/2
lx_g2=np.linspace(0.05,3,100);lz_g2=np.linspace(0.1,0.6,100)
LX2,LZ2=np.meshgrid(lx_g2,lz_g2)
sh_in2=np.column_stack([np.full_like(LX2.ravel(),xcB),np.full_like(LX2.ravel(),zcB),LX2.ravel(),LZ2.ravel()])
sh_out2=eval_phys(sh_in2,batch=500).reshape(100,100)

fig3,axes3=plt.subplots(1,3,figsize=(18,5))
ax=axes3[0]
cs2=ax.contourf(LX2,LZ2,sh_out2*100,levels=30,cmap='RdYlGn_r')
ax.plot(A_B[2],A_B[3],'ro',markersize=12,markeredgewidth=2,label=f'A (Lx={A_B[2]:.3f},Lz={A_B[3]:.2f})')
ax.plot(B_B[2],B_B[3],'bD',markersize=10,markeredgewidth=2,label=f'B (Lx={B_B[2]:.3f},Lz={B_B[3]:.2f})')
ax.contour(LX2,LZ2,sh_out2*100,levels=[10],colors='red',linewidths=2,linestyles='--')
plt.colorbar(cs2,ax=ax,label='Outage [%]');ax.set_xlabel('Lx [m]');ax.set_ylabel('Lz [m]')
ax.set_title(f'Jump B: (Lx,Lz) at xc={xcB:.1f},zc={zcB:.2f}');ax.legend();ax.grid(alpha=0.2)

ax=axes3[1]
ax.barh(['A'],[A_A[3]],height=0.08,left=A_A[0]-A_A[2]/2,color='red',alpha=0.5,edgecolor='black')
ax.barh(['B'],[B_A[3]],height=0.08,left=B_A[0]-B_A[2]/2,color='blue',alpha=0.5,edgecolor='black')
ax.annotate(f'Lz={A_A[3]:.2f}m',(A_A[0],0.1),ha='center',fontsize=9,color='red',fontweight='bold')
ax.annotate(f'Lz={B_A[3]:.2f}m',(B_A[0],0.3),ha='center',fontsize=9,color='blue',fontweight='bold')
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.set_xlabel('x [m]');ax.set_ylabel('')
ax.set_title(f'Jump A: Lz {A_A[3]:.2f}→{B_A[3]:.2f}m (—{abs(A_A[3]-B_A[3])/A_A[3]*100:.0f}%) 十字転置');ax.grid(alpha=0.3)

ax=axes3[2]
ax.barh(['A'],[A_B[3]],height=0.08,left=A_B[0]-A_B[2]/2,color='red',alpha=0.5,edgecolor='black')
ax.barh(['B'],[B_B[3]],height=0.08,left=B_B[0]-B_B[2]/2,color='blue',alpha=0.5,edgecolor='black')
ax.set_xlim(0,10);ax.set_ylim(0,2);ax.set_xlabel('x [m]');ax.set_ylabel('')
ax.set_title(f'Jump B: Lx {A_B[2]:.2f}→{B_B[2]:.2f}m (—{abs(A_B[2]-B_B[2])/A_B[2]*100:.0f}%) 同向压缩');ax.grid(alpha=0.3)

plt.tight_layout();plt.savefig('fig3_jumpB.png',dpi=150);plt.show()

# ============================================================
# Diagnosis
# ============================================================
print('\n'+'='*70)
print('DIAGNOSIS')
print('='*70)
dLA=np.abs(A_A[2]-B_A[2]);dWA=np.abs(A_A[3]-B_A[3])
print(f'\nJump A @ {pf[jA,1]*100:.1f}% → TRUE 十字転置')
print(f'  Lz: {A_A[3]:.2f}→{B_A[3]:.2f} m  (—{dWA/A_A[3]*100:.0f}%)  ← 突变轴')
print(f'  Lx: {A_A[2]:.1f}→{B_A[2]:.1f} m  (+{dLA/min(A_A[2],B_A[2])*100:.0f}%)  ← 补偿拉伸')
print(f'  Lz 骤降 60% → 垂直衍射角扩大 2.5 倍 → 模式质变')

dLB=np.abs(A_B[2]-B_B[2]);dWB=np.abs(A_B[3]-B_B[3])
print(f'\nJump B @ {pf[jB,1]*100:.1f}% → 同向极限压缩')
print(f'  Lx: {A_B[2]:.3f}→{B_B[2]:.3f} m  (—{dLB/A_B[2]*100:.0f}%)  ← 突变轴')
print(f'  Lz: {A_B[3]:.3f}→{B_B[3]:.3f} m  (+{dWB/max(A_B[3],1e-9)*100:.0f}%)  ← 冻结')
print(f'  Lx 压缩到衍射极限，Lz 从未切换物理策略。')
print(f'  Aspect 翻转 (8.7→0.7) 是数学副产物，不是模式切换。')
print('='*70)

from IPython.display import FileLink,display
for f in ['fig1_tracking.png','fig2_jumpA.png','fig3_jumpB.png']:
    display(FileLink(f))